# Prior Auth Assist — Patient Data Analysis

Queries the live HAPI FHIR R4 server and analyzes the demo + Synthea patient population.
Useful for:
- Verifying demo patients loaded correctly
- Exploring the Synthea patient population
- Visualizing prior-auth compliance gaps
- Sanity-checking rule criteria thresholds

**Setup:** Make sure the FHIR server is reachable. In-cluster:
```bash
kubectl port-forward -n priorauth-fhir svc/priorauth-fhir 8080:8080
```
Then set `FHIR_BASE_URL` below.

In [ ]:
import os
import requests
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from datetime import datetime, timedelta
from collections import defaultdict
import json
import warnings
warnings.filterwarnings('ignore')

FHIR_BASE = os.getenv('FHIR_BASE_URL', 'http://localhost:8080/fhir')
HEADERS = {'Accept': 'application/fhir+json'}

print(f'FHIR server: {FHIR_BASE}')

# Verify connectivity
try:
    r = requests.get(f'{FHIR_BASE}/metadata', headers=HEADERS, timeout=10)
    cap = r.json()
    print(f'Connected ✓  FHIR version: {cap.get("fhirVersion")}  Software: {cap.get("software", {}).get("name")}')
except Exception as e:
    print(f'ERROR connecting to FHIR server: {e}')
    print('Run: kubectl port-forward -n priorauth-fhir svc/priorauth-fhir 8080:8080')

## Helper — paginated FHIR search

In [ ]:
def fhir_search_all(resource_type, params=None, max_pages=20):
    """Fetch all pages of a FHIR search, return list of resources."""
    url = f'{FHIR_BASE}/{resource_type}'
    p = dict(params or {})
    p.setdefault('_count', '100')
    results = []
    pages = 0
    while url and pages < max_pages:
        r = requests.get(url, params=p if pages == 0 else None, headers=HEADERS, timeout=30)
        r.raise_for_status()
        bundle = r.json()
        for entry in bundle.get('entry', []):
            results.append(entry['resource'])
        # Follow next link
        url = next((lnk['url'] for lnk in bundle.get('link', []) if lnk['relation'] == 'next'), None)
        pages += 1
    return results


def extract_name(patient):
    names = patient.get('name', [])
    if names:
        n = names[0]
        family = n.get('family', '')
        given = ' '.join(n.get('given', []))
        return f'{given} {family}'.strip()
    return patient.get('id', 'Unknown')


def age_from_dob(dob_str):
    if not dob_str:
        return None
    try:
        dob = datetime.strptime(dob_str[:10], '%Y-%m-%d')
        return (datetime.now() - dob).days // 365
    except:
        return None


def get_codes(coding_list):
    return [(c.get('system',''), c.get('code',''), c.get('display','')) for c in (coding_list or [])]

print('Helpers loaded.')

## 1. Patient Population Overview

In [ ]:
print('Fetching all patients...')
patients = fhir_search_all('Patient')
print(f'Total patients: {len(patients)}')

rows = []
for p in patients:
    rows.append({
        'id': p.get('id'),
        'name': extract_name(p),
        'gender': p.get('gender', 'unknown'),
        'dob': p.get('birthDate'),
        'age': age_from_dob(p.get('birthDate')),
        'is_demo': p.get('id', '').endswith('-001'),
    })

df_patients = pd.DataFrame(rows)
print(f"\nDemo patients: {df_patients['is_demo'].sum()}")
print(f"Synthea patients: {(~df_patients['is_demo']).sum()}")
df_patients.head(10)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
fig.suptitle('Patient Population — Demographics', fontsize=14)

# Age distribution
ax = axes[0]
df_ages = df_patients['age'].dropna()
ax.hist(df_ages, bins=20, color='steelblue', edgecolor='white')
ax.set_title('Age Distribution')
ax.set_xlabel('Age (years)')
ax.set_ylabel('Count')
ax.axvline(df_ages.median(), color='orange', linestyle='--', label=f'Median: {df_ages.median():.0f}')
ax.legend()

# Gender split
ax = axes[1]
gender_counts = df_patients['gender'].value_counts()
colors = {'male': 'steelblue', 'female': 'salmon', 'unknown': 'grey'}
ax.pie(gender_counts.values, labels=gender_counts.index,
       colors=[colors.get(g, 'lightgrey') for g in gender_counts.index],
       autopct='%1.0f%%', startangle=90)
ax.set_title('Gender Split')

# Demo vs Synthea
ax = axes[2]
cat_counts = df_patients['is_demo'].map({True: 'Demo', False: 'Synthea'}).value_counts()
ax.bar(cat_counts.index, cat_counts.values, color=['gold', 'steelblue'])
ax.set_title('Patient Source')
ax.set_ylabel('Count')
for i, v in enumerate(cat_counts.values):
    ax.text(i, v + 0.3, str(v), ha='center', fontweight='bold')

plt.tight_layout()
plt.show()

## 2. Demo Patients — Verify Load

In [ ]:
DEMO_PATIENT_IDS = [
    'knee-complete-001',
    'knee-partial-001',
    'lumbar-missing-001',
    'sleep-complete-001',
    'drug-step-met-001',
]

DEMO_DESCRIPTIONS = {
    'knee-complete-001':   'Anna Kowalski — Knee MRI (all documented)',
    'knee-partial-001':    'Marcus Webb — Knee MRI (only 8/12 PT visits)',
    'lumbar-missing-001':  'Carol Simmons — Lumbar MRI (no PT at all)',
    'sleep-complete-001':  'Robert Nguyen — Sleep Study (all documented)',
    'drug-step-met-001':   'Elena Vasquez — Adalimumab (step therapy met)',
}

print('Demo Patient Verification')
print('=' * 60)
for pid in DEMO_PATIENT_IDS:
    r = requests.get(f'{FHIR_BASE}/Patient/{pid}', headers=HEADERS)
    if r.status_code == 200:
        p = r.json()
        print(f'  ✓  {pid}: {extract_name(p)} ({p.get("gender")}, DOB {p.get("birthDate")})')
    else:
        print(f'  ✗  {pid}: NOT FOUND (HTTP {r.status_code}) — run load-data.sh')

## 3. Demo Patient — Resource Counts

In [ ]:
def count_patient_resources(patient_id):
    counts = {}
    for rtype in ['Condition', 'Observation', 'Procedure', 'MedicationRequest']:
        resources = fhir_search_all(rtype, {'patient': patient_id})
        counts[rtype] = len(resources)
    return counts

print('Counting resources for demo patients...\n')
resource_data = []
for pid in DEMO_PATIENT_IDS:
    counts = count_patient_resources(pid)
    row = {'patient': pid.replace('-001', ''), **counts}
    resource_data.append(row)
    print(f'  {pid}: {counts}')

df_resources = pd.DataFrame(resource_data).set_index('patient')
print()
df_resources

In [ ]:
ax = df_resources.plot(kind='bar', figsize=(12, 5), colormap='Set2', edgecolor='white')
ax.set_title('Resource Counts per Demo Patient', fontsize=13)
ax.set_xlabel('Patient')
ax.set_ylabel('Count')
ax.set_xticklabels(ax.get_xticklabels(), rotation=20, ha='right')
ax.legend(title='Resource Type')
plt.tight_layout()
plt.show()

## 4. Knee MRI — PT Visit Timeline (Anna vs Marcus)

In [ ]:
PT_SNOMED = '229070002'
PT_SYSTEM = 'http://snomed.info/sct'

def get_pt_procedures(patient_id):
    procs = fhir_search_all('Procedure', {'patient': patient_id, 'status': 'completed'})
    pt_visits = []
    for proc in procs:
        for coding in proc.get('code', {}).get('coding', []):
            if coding.get('system') == PT_SYSTEM and coding.get('code') == PT_SNOMED:
                date_str = proc.get('performedDateTime') or proc.get('performedPeriod', {}).get('start', '')
                if date_str:
                    pt_visits.append(datetime.strptime(date_str[:10], '%Y-%m-%d'))
    return sorted(pt_visits)

anna_pt = get_pt_procedures('knee-complete-001')
marcus_pt = get_pt_procedures('knee-partial-001')

print(f'Anna Kowalski (knee-complete) — PT visits: {len(anna_pt)}')
print(f'Marcus Webb (knee-partial)   — PT visits: {len(marcus_pt)}')

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(12, 5), sharex=False)
fig.suptitle('Physical Therapy Visit Timeline\n(Knee MRI requires ≥12 visits in 90 days)', fontsize=13)

THRESHOLD = 12

for ax, visits, name, color in [
    (axes[0], anna_pt,   'Anna Kowalski (complete: 14 visits)',  'steelblue'),
    (axes[1], marcus_pt, 'Marcus Webb (partial: 8 visits)',       'salmon'),
]:
    if visits:
        ax.scatter(visits, [1]*len(visits), s=120, color=color, zorder=5)
        for d in visits:
            ax.axvline(d, color=color, alpha=0.3, linewidth=1)
    status = '✓ DOCUMENTED' if len(visits) >= THRESHOLD else f'⚠ PARTIAL ({len(visits)}/{THRESHOLD})'
    ax.set_title(f'{name}  →  {status}', fontsize=11)
    ax.set_yticks([])
    ax.xaxis.set_major_formatter(plt.matplotlib.dates.DateFormatter('%b %d'))
    ax.annotate(f'{len(visits)} visits', xy=(0.02, 0.6), xycoords='axes fraction',
                fontsize=12, fontweight='bold', color=color)

plt.tight_layout()
plt.show()

## 5. Rule Compliance Heatmap — All Demo Patients

In [ ]:
# Call the MCP server's prefill_documentation via HTTP transport
# (requires MCP server running at localhost:8080/mcp or set MCP_BASE below)
# If the MCP server is not running, this cell is skipped gracefully.

MCP_BASE = os.getenv('MCP_BASE_URL', 'http://localhost:8080')

SCENARIOS = [
    ('knee-complete-001',  'Knee MRI'),
    ('knee-partial-001',   'Knee MRI'),
    ('lumbar-missing-001', 'Lumbar MRI'),
    ('sleep-complete-001', 'Sleep Study'),
    ('drug-step-met-001',  'Adalimumab (Humira)'),
]

STATUS_ORDER = ['DOCUMENTED', 'PARTIAL', 'MISSING']
STATUS_COLOR = {'DOCUMENTED': 2, 'PARTIAL': 1, 'MISSING': 0}

compliance_rows = []
mcp_available = False

try:
    r = requests.get(f'{MCP_BASE}/actuator/health', timeout=3)
    mcp_available = r.status_code == 200
except:
    pass

if mcp_available:
    print(f'MCP server reachable at {MCP_BASE}')
    for patient_id, service in SCENARIOS:
        # Call via MCP HTTP/SSE transport is complex; instead call the REST endpoint
        # For this notebook we replicate the logic directly using FHIR data
        print(f'  Evaluating {patient_id} / {service}...')
else:
    print(f'MCP server not reachable ({MCP_BASE}). Performing direct FHIR analysis instead.')
    print('Start the MCP server and re-run to use the MCP transport.')

In [ ]:
# Direct FHIR evaluation — mirrors the Java RuleEvaluator logic for the notebook
# This is intentionally independent code for data-science exploration.

def check_observation_count(patient_id, snomed_system, code, min_occurrences, lookback_days):
    cutoff = (datetime.now() - timedelta(days=lookback_days)).strftime('%Y-%m-%d')
    obs = fhir_search_all('Procedure', {'patient': patient_id})
    count = 0
    for o in obs:
        for c in o.get('code', {}).get('coding', []):
            if c.get('system') == snomed_system and c.get('code') == code:
                date_str = (o.get('performedDateTime') or
                            o.get('performedPeriod', {}).get('start', ''))
                if date_str and date_str[:10] >= cutoff:
                    count += 1
    if count >= min_occurrences:
        return 'DOCUMENTED'
    elif count > 0:
        return 'PARTIAL'
    return 'MISSING'


def check_procedure_present(patient_id, codes):
    procs = fhir_search_all('Procedure', {'patient': patient_id, 'status': 'completed'})
    for proc in procs:
        for c in proc.get('code', {}).get('coding', []):
            if c.get('code') in codes:
                return 'DOCUMENTED'
    return 'MISSING'


def check_observation_value(patient_id, loinc_code, min_value):
    obs = fhir_search_all('Observation', {'patient': patient_id,
                                          'code': f'http://loinc.org|{loinc_code}'})
    for o in obs:
        val = o.get('valueQuantity', {}).get('value')
        if val is not None and float(val) >= min_value:
            return 'DOCUMENTED'
        elif val is not None:
            return 'PARTIAL'
    return 'MISSING'


def check_condition_present(patient_id, codes):
    conditions = fhir_search_all('Condition', {'patient': patient_id, 'clinical-status': 'active'})
    for cond in conditions:
        for c in cond.get('code', {}).get('coding', []):
            if c.get('code') in codes:
                return 'DOCUMENTED'
    return 'MISSING'


def check_medication_present(patient_id, codes):
    meds = fhir_search_all('MedicationRequest', {'patient': patient_id})
    for med in meds:
        med_codes = (med.get('medicationCodeableConcept', {}).get('coding', []) or
                     med.get('medication', {}).get('concept', {}).get('coding', []))
        for c in med_codes:
            if c.get('code') in codes:
                return 'DOCUMENTED'
    return 'MISSING'


# Evaluate all demo scenarios
PT_SNOMED_CODES = ('http://snomed.info/sct', '229070002')
XRAY_KNEE_CODES = ['24727-8', '24728-6', '36643-5']
XRAY_LUMBAR_CODES = ['24969-6', '36643-5']
PAIN_LOINC = '72514-3'
EPWORTH_LOINC = '89243-0'
BMI_LOINC = '39156-5'
SLEEP_SNOMED = ['72863001', '73430006', '44186003']
RA_ICD = ['M05.79', 'M06.09', 'M05.9', 'M06.9']
DMARD_RX = ['41493', '5521', '9524', '35617']

evaluation_results = []

print('Evaluating demo patients against coverage rules...')

# Knee MRI — Anna (complete)
pid = 'knee-complete-001'
evaluation_results += [
    (pid, 'Knee MRI', 'pt_duration',  check_observation_count(pid, *PT_SNOMED_CODES, 12, 90)),
    (pid, 'Knee MRI', 'prior_xray',   check_procedure_present(pid, XRAY_KNEE_CODES)),
    (pid, 'Knee MRI', 'pain_score',   check_observation_value(pid, PAIN_LOINC, 4.0)),
]

# Knee MRI — Marcus (partial)
pid = 'knee-partial-001'
evaluation_results += [
    (pid, 'Knee MRI', 'pt_duration',  check_observation_count(pid, *PT_SNOMED_CODES, 12, 90)),
    (pid, 'Knee MRI', 'prior_xray',   check_procedure_present(pid, XRAY_KNEE_CODES)),
    (pid, 'Knee MRI', 'pain_score',   check_observation_value(pid, PAIN_LOINC, 4.0)),
]

# Lumbar MRI — Carol (missing PT)
pid = 'lumbar-missing-001'
evaluation_results += [
    (pid, 'Lumbar MRI', 'pt_duration', check_observation_count(pid, *PT_SNOMED_CODES, 12, 90)),
    (pid, 'Lumbar MRI', 'prior_xray',  check_procedure_present(pid, XRAY_LUMBAR_CODES)),
    (pid, 'Lumbar MRI', 'pain_score',  check_observation_value(pid, PAIN_LOINC, 4.0)),
]

# Sleep Study — Robert (complete)
pid = 'sleep-complete-001'
evaluation_results += [
    (pid, 'Sleep Study', 'epworth_score',    check_observation_value(pid, EPWORTH_LOINC, 10.0)),
    (pid, 'Sleep Study', 'bmi_documented',   check_observation_count(pid, 'http://loinc.org', BMI_LOINC, 1, 180)),
    (pid, 'Sleep Study', 'sleep_symptom',    check_condition_present(pid, SLEEP_SNOMED)),
]

# Adalimumab — Elena (step met)
pid = 'drug-step-met-001'
evaluation_results += [
    (pid, 'Adalimumab', 'ra_diagnosis',  check_condition_present(pid, RA_ICD)),
    (pid, 'Adalimumab', 'prior_dmard_1', check_medication_present(pid, DMARD_RX)),
    (pid, 'Adalimumab', 'prior_dmard_2', check_medication_present(pid, DMARD_RX)),
]

df_eval = pd.DataFrame(evaluation_results, columns=['patient_id', 'service', 'criterion', 'status'])
df_eval['patient_short'] = df_eval['patient_id'].str.replace('-001', '').str.replace('-', ' ').str.title()
print('Done.')
df_eval

In [ ]:
# Build pivot for heatmap
df_eval['label'] = df_eval['patient_short'] + '\n(' + df_eval['service'] + ')'
df_pivot = df_eval.pivot(index='label', columns='criterion', values='status')

# Map status to numeric
status_num = df_pivot.applymap(lambda s: STATUS_COLOR.get(s, -1) if isinstance(s, str) else -1)

fig, ax = plt.subplots(figsize=(13, 6))
cmap = plt.cm.get_cmap('RdYlGn', 3)
im = ax.imshow(status_num.values, aspect='auto', cmap=cmap, vmin=-0.5, vmax=2.5)

ax.set_xticks(range(len(df_pivot.columns)))
ax.set_xticklabels(df_pivot.columns, rotation=30, ha='right', fontsize=10)
ax.set_yticks(range(len(df_pivot.index)))
ax.set_yticklabels(df_pivot.index, fontsize=10)

# Annotate cells
for i in range(len(df_pivot.index)):
    for j in range(len(df_pivot.columns)):
        val = df_pivot.values[i, j]
        if isinstance(val, str):
            short = {'DOCUMENTED': '✓', 'PARTIAL': '~', 'MISSING': '✗'}.get(val, val)
            ax.text(j, i, short, ha='center', va='center', fontsize=14, fontweight='bold',
                    color='white' if val == 'MISSING' else 'black')

# Legend
patches = [
    mpatches.Patch(color=cmap(2), label='DOCUMENTED'),
    mpatches.Patch(color=cmap(1), label='PARTIAL'),
    mpatches.Patch(color=cmap(0), label='MISSING'),
]
ax.legend(handles=patches, loc='upper right', bbox_to_anchor=(1.18, 1.0))
ax.set_title('Prior Auth Compliance — Demo Patients × Criteria', fontsize=13, pad=15)
plt.tight_layout()
plt.show()

## 6. Pain Score Distribution Across All Patients

In [ ]:
print('Fetching pain score observations (LOINC 72514-3) across all patients...')
pain_obs = fhir_search_all('Observation', {
    'code': 'http://loinc.org|72514-3',
    '_count': '200'
})
print(f'Found {len(pain_obs)} pain observations')

pain_scores = []
for o in pain_obs:
    val = o.get('valueQuantity', {}).get('value')
    patient_ref = o.get('subject', {}).get('reference', '')
    pid = patient_ref.split('/')[-1] if patient_ref else 'unknown'
    is_demo = pid.endswith('-001')
    if val is not None:
        pain_scores.append({'patient_id': pid, 'score': float(val), 'is_demo': is_demo})

df_pain = pd.DataFrame(pain_scores)
print(f'Parseable pain scores: {len(df_pain)}')
df_pain.head()

In [ ]:
if not df_pain.empty:
    fig, ax = plt.subplots(figsize=(10, 4))
    ax.hist(df_pain[~df_pain['is_demo']]['score'], bins=10, range=(0, 10),
            label='Synthea patients', color='steelblue', alpha=0.7, edgecolor='white')
    if df_pain['is_demo'].any():
        ax.hist(df_pain[df_pain['is_demo']]['score'], bins=10, range=(0, 10),
                label='Demo patients', color='gold', alpha=0.9, edgecolor='black', linewidth=0.8)
    ax.axvline(4, color='red', linestyle='--', linewidth=1.5, label='PA threshold (>4)')
    ax.set_title('Pain Score Distribution (LOINC 72514-3)', fontsize=12)
    ax.set_xlabel('Pain Score (0–10)')
    ax.set_ylabel('Count')
    ax.set_xticks(range(0, 11))
    ax.legend()
    plt.tight_layout()
    plt.show()
    
    above_threshold = (df_pain['score'] > 4).sum()
    pct = 100 * above_threshold / len(df_pain)
    print(f'Patients with pain > 4/10: {above_threshold}/{len(df_pain)} ({pct:.1f}%) — would meet pain criterion')
else:
    print('No pain score observations found. Load demo patients and/or Synthea data first.')

## 7. Condition Frequency (Top Diagnoses in Population)

In [ ]:
print('Fetching all active conditions...')
conditions = fhir_search_all('Condition', {'clinical-status': 'active', '_count': '500'}, max_pages=10)
print(f'Found {len(conditions)} active conditions')

code_counts = defaultdict(lambda: {'count': 0, 'display': ''})
for cond in conditions:
    for coding in cond.get('code', {}).get('coding', []):
        code = coding.get('code', '')
        display = coding.get('display', code)
        code_counts[code]['count'] += 1
        if not code_counts[code]['display']:
            code_counts[code]['display'] = display

df_cond = pd.DataFrame([
    {'code': k, 'display': v['display'][:45], 'count': v['count']}
    for k, v in code_counts.items()
]).sort_values('count', ascending=False).head(20)

fig, ax = plt.subplots(figsize=(12, 6))
bars = ax.barh(range(len(df_cond)), df_cond['count'], color='steelblue', edgecolor='white')
ax.set_yticks(range(len(df_cond)))
ax.set_yticklabels([f"{r['code']}: {r['display']}" for _, r in df_cond.iterrows()], fontsize=9)
ax.invert_yaxis()
ax.set_title('Top 20 Active Conditions in Patient Population', fontsize=12)
ax.set_xlabel('Count')
for bar, val in zip(bars, df_cond['count']):
    ax.text(bar.get_width() + 0.1, bar.get_y() + bar.get_height()/2,
            str(val), va='center', fontsize=8)
plt.tight_layout()
plt.show()

## 8. Prior Auth Coverage Gap Summary

In [ ]:
if not df_eval.empty:
    summary = df_eval.groupby('status').size().reindex(['DOCUMENTED', 'PARTIAL', 'MISSING'], fill_value=0)
    total = summary.sum()

    fig, axes = plt.subplots(1, 2, figsize=(12, 5))
    fig.suptitle('Prior Auth Readiness — Demo Patients', fontsize=13)

    # Overall pie
    ax = axes[0]
    colors = ['#4caf50', '#ff9800', '#f44336']
    wedges, texts, autotexts = ax.pie(
        summary.values, labels=summary.index, colors=colors,
        autopct='%1.0f%%', startangle=90
    )
    ax.set_title(f'All Criteria ({total} total)')

    # Per-patient stacked bar
    ax = axes[1]
    pivot = df_eval.groupby(['patient_short', 'status']).size().unstack(fill_value=0)
    for col in ['DOCUMENTED', 'PARTIAL', 'MISSING']:
        if col not in pivot.columns:
            pivot[col] = 0
    pivot = pivot[['DOCUMENTED', 'PARTIAL', 'MISSING']]
    pivot.plot(kind='bar', stacked=True, ax=ax,
               color=['#4caf50', '#ff9800', '#f44336'], edgecolor='white')
    ax.set_title('Criteria Status per Patient')
    ax.set_xlabel('')
    ax.set_ylabel('Criterion Count')
    ax.set_xticklabels(ax.get_xticklabels(), rotation=25, ha='right', fontsize=9)
    ax.legend(title='Status')

    plt.tight_layout()
    plt.show()

    print('\nGap analysis:')
    missing = df_eval[df_eval['status'].isin(['MISSING', 'PARTIAL'])]
    for _, row in missing.iterrows():
        print(f'  [{row.status:12s}]  {row.patient_short:30s}  {row.criterion}')

---

## Notes

This notebook evaluates criteria using direct FHIR queries — the same logic as the Java `RuleEvaluator`.
It is intentionally independent of the MCP server for data-science exploration.

To call the MCP server directly from Python, use the `mcp` Python client:
```python
from mcp import ClientSession
from mcp.client.streamable_http import streamablehttp_client

async with streamablehttp_client('http://localhost:8080/mcp') as (r, w, _):
    async with ClientSession(r, w) as session:
        await session.initialize()
        result = await session.call_tool('find_patient', {'query': 'Anna Kowalski'})
        print(result.content[0].text)
```